In [3]:
import xarray as xr
from daz_iono import *
from lics_unwrap import *
from scipy.constants import speed_of_light
import numpy as np
from scipy.interpolate import griddata
from LiCSAR_misc import *
import framecare as fc
from scipy.interpolate import interp1d

def get_inc_frame(frame, heading=False):
    '''will get the incidence angle 2d xr.datarray
    if heading==True: return also heading raster
    '''
    metadir = os.path.join(os.environ['LiCSAR_public'], str(int(frame[:3])), frame, 'metadata')
    Ufile = os.path.join(metadir, frame + '.geo.U.tif')
    U = load_tif2xr(Ufile)
    U = U.where(U != 0)
    inc = U.copy()
    inc.values = np.degrees(np.arccos(U.values))
    if heading:
        Efile = os.path.join(metadir, frame + '.geo.E.tif')
        Nfile = os.path.join(metadir, frame + '.geo.N.tif')
        E = load_tif2xr(Efile)
        N = load_tif2xr(Nfile)
        E = E.where(E != 0)
        N = N.where(N != 0)
        heading = N.copy(deep=True)
        heading.values = np.degrees(np.arctan2(E, N)) + 90
        return inc, heading
    else:
        return inc



In [4]:
def get_vtec_from_code(acqtime, lat = 0, lon = 0, storedir = '/gws/nopw/j04/nceo_geohazards_vol1/code_iono', return_fullxr = False):
    """ Adapted from Reza Bordbari script, plus using functions from https://notebook.community/daniestevez/jupyter_notebooks/IONEX
    
    17/03/2025-(MN):function also helps to extract NASA JPL High Resolution vTEC values (15min, 1x1degree) at first. 
    https://sideshow.jpl.nasa.gov/pub/iono_daily/gim_for_research/jpli/
    
    Args:
        acqtime (dt.datetime)
        lat (float)
        lon (float)
        storedir (str)
        return_fullxr (bool): if True, will return full TEC datacube
    """
    #D = acqtime.strftime('%Y%m%d')
    #ipp = np.array([lat,lon])
    # check if exists:
    fna = glob.glob(storedir + '/jpld' + acqtime.strftime('%j') + '0.' + acqtime.strftime('%y') + '*.nc') ##priotarize JPL-HR GIM, this can be shrink in the future.
    if fna:  
        ionix = os.path.join(storedir, fna[0])  # Found JPL-HR GIM file, use it
    else:
        # JPL-HR GIM does not exist, try to download it.
        ionix = download_code_data(acqtime, storedir)
        fna = glob.glob(storedir + '/jpld' + acqtime.strftime('%j') + '0.' + acqtime.strftime('%y') + '*.nc') ##priotarize JPL-HR GIM
    
        if fna:  
            ionix = os.path.join(storedir, fna[0])  # Found a different GIM file, use it
        else:
            # If JPL-HR GIM is still missing, fallback to CODE GIM
            fna = glob.glob(storedir + '/????' + acqtime.strftime('%j') + '0.' + acqtime.strftime('%y') + '?') #CODE GIM
            if fna:
                ionix = os.path.join(storedir, fna[0])
            else:
                #If no CODE GIM is found, try to download it
                ionix = download_code_data(acqtime, storedir)
    if not ionix:
        return False
    #else:
    #    rc=os.system('rm '+fullpath) # clean the .Z
    # prep 
    #hhmmss=acqtime.strftime('%H%M%S')
    # loading the TEC maps, thanks to https://notebook.community/daniestevez/jupyter_notebooks/IONEX (but improved towards xarray by ML B-)
    if os.path.basename(ionix).startswith('jpl'):
        # Open the NetCDF file
        ds = xr.open_dataset(ionix)
        # Convert time epochs to readable datetime format, 15min resolution referenced to j2000 (1/1/2000 12:00 UT).
        time_values = ds['time'].values
        #converting yyyy-mm-ddThh:mm:ss format. #TODO: We can change that regarding ML's wishes. I found this is clear for now.
        converted_times = pd.to_datetime(time_values, origin='2000-01-01 12:00:00', unit='s')
        #dataset2dataarray
        tecxr_jhr = xr.DataArray(data=ds['tecmap'].values, dims=['time','lat','lon'],
                            coords=dict(time=converted_times, lat= ds["lat"].values, lon= ds["lon"].values) )
        tecxr=tecxr_jhr*1e+16 # from TECU   
    else:
        try:
            tecmaps = get_tecmaps(ionix)
        except:
            print('ERROR loading ionix file: '+ionix)
            return False
        try:
            interval = int(grep1line('INTERVAL',ionix).split()[0])
        except:
            print('ERROR, the ionix file '+ionix+' does not contain necessary keywords. Cancelling')
            return False
        timestep = interval/3600
        timecoords = np.arange(0.0,24.0+timestep,timestep)  # we expect start/end time being midnight, should be standard for all CODE files?
        lat_all = np.arange(87.5,-87.5-2.5,-2.5)
        lon_all = np.arange(-180.0,180.0+5,5.0)
        tecxr = xr.DataArray(data=tecmaps, dims=['time','lat','lon'],
                            coords=dict(time=timecoords, lon=lon_all, lat=lat_all) )
        # interpolate through the nan values
        tonan=9999
        tecxr.where(tecxr!=tonan)
        tecxr=tecxr.interpolate_na(dim="lon", method="linear", fill_value="extrapolate")
        tecxr = tecxr*1e+16 # from TECU
    if return_fullxr:
        return tecxr
    else:
        return get_vtec_from_tecxr(tecxr, acqtime, lat, lon)

In [5]:
def download_code_data(acqtime, storedir = '/gws/nopw/j04/nceo_geohazards_vol1/code_iono'):
    """Downloads Ionospheric TEC data from JPL or CODE."""
    ffound = False
    if not ffound:
        filename = 'jpld'+ acqtime.strftime('%j') + '0.' + acqtime.strftime('%y')+'i.nc.gz' # TODO: check YMD
        url = 'https://sideshow.jpl.nasa.gov/pub/iono_daily/gim_for_research/jpld/' + str(acqtime.year) + '/' + filename
        fullpath = os.path.join(storedir, filename)
        ionix = fullpath[:-3]
        if not os.path.exists(ionix):
            if not os.path.exists(fullpath):
                # download this
                try:
                    wget.download(url, out=storedir)
                    ffound = True 
                except:
                    #print('jpl-hr not exist')
                    ffound = False
            if os.path.exists(fullpath):
                ffound = True
        else:
            ffound = True
    ###if JPL HR-GIM is not available, then try to download CODE data. After, 2024-08-01 we need CODE data. 
    if not ffound:  
        instrings = ['CODG', 'CGIM']
        #lastrings = ['I', 'N']
        for instr in instrings:
            #for lastr in lastrings:
            filename = instr + acqtime.strftime('%j') + '0.' + acqtime.strftime('%y') + 'I.Z'
            url = 'http://ftp.aiub.unibe.ch/CODE/' + acqtime.strftime('%Y') + '/' + filename
            fullpath = os.path.join(storedir, filename)
            ionix = fullpath[:-2]
            if not os.path.exists(ionix):
                if not os.path.exists(fullpath):
                    # download this
                    try:
                        wget.download(url, out=storedir)
                        ffound = True 
                    except:
                        #print('code-oldname not exists')
                        ffound = False
                if os.path.exists(fullpath):
                    ffound = True
            else:
                ffound = True
            if ffound:
                break  #Exit loop if successful download
    # since 12/2022 they changed naming convention to e.g. COD0OPSFIN_20230510000_01D_01H_GIM.INX.gz
    # see https://cddis.nasa.gov/Data_and_Derived_Products/GNSS/atmospheric_products.html#iono
    if not ffound:
        filename = 'COD0OPSFIN_'+ acqtime.strftime('%Y') + acqtime.strftime('%j')+'0000_01D_01H_GIM.INX.gz' # TODO: check YMD
        #filename = instr + acqtime.strftime('%j') + '0.' + acqtime.strftime('%y') + 'I.Z'
        url = 'http://ftp.aiub.unibe.ch/CODE/' + acqtime.strftime('%Y') + '/' + filename
        fullpath = os.path.join(storedir, filename)
        ionix = fullpath[:-3]
        if not os.path.exists(ionix):
            if not os.path.exists(fullpath):
                # download this
                try:
                    wgotfile = wget.download(url, out=storedir)
                except:
                    print('error during wget download')
                    ffound = False
            if os.path.exists(fullpath):
                ffound = True
        else:
            ffound = True
    if not ffound:
        print('no CODE layer found for '+filename)
        return False
    if not os.path.exists(ionix):
        rc = os.system('cd ' + storedir + '; 7za x ' + filename + ' >/dev/null 2>/dev/null; rm ' + fullpath)
    if not os.path.exists(ionix):
        print('ERROR: maybe you do not have 7za installed')
        return False
    return ionix

In [6]:
def get_vtec_from_tecxr(tecxr, acqtime, lat, lon, rotate=True):
    if len(tecxr.time.values) == 25:   
        h_time = float(acqtime.strftime('%H'))
        m_time = float(acqtime.strftime('%M'))
        s_time = float(acqtime.strftime('%S'))
        # given time in decimal format
        time_dec = h_time + (m_time/60) + (s_time / 3600)
        # ML: 2023/08, based on : https://github.com/insarlab/MintPy/blob/main/src/mintpy/objects/ionex.py
        # that is actually based on
        # Schaer, S., Gurtner, W., & Feltens, J. (1998). IONEX: The ionosphere map exchange format
        #         version 1.1. Paper presented at the Proceedings of the IGS AC workshop, Darmstadt, Germany.
        if rotate:
            # 3D interpolation with rotation as above reference
            try:
                htimes = tecxr.time.values
            except:
                htimes = np.array([float(tecxr.time)])  #in case of only one value
            pretime = int(htimes[htimes <= time_dec][-1])
            postime = int(htimes[htimes >= time_dec][0])
            #
            lon0 = lon + (time_dec - pretime) * 360. / 24.
            lon1 = lon + (time_dec - postime) * 360. / 24.
            print(time_dec - pretime)
            print(time_dec - postime)
            #
            tec_val0 = float(tecxr.interp(time=pretime, lon=lon0, lat=lat, method='linear'))
            tec_val1 = float(tecxr.interp(time=postime, lon=lon1, lat=lat, method='linear'))
            #
            tec = ((postime - time_dec) / (postime - pretime) * tec_val0
                       + (time_dec - pretime) / (postime - pretime) * tec_val1)
        else:
            # previous attempt, but still too different from the S1_ETAD CODE outputs (that rotates the Earth towards the Sun..)
            tec = float(tecxr.interp(time=time_dec, lon=lon,lat=lat, method='cubic')) # should be better than linear, but maybe quadratic is more suitable?
    elif len(tecxr.time.values) == 96:
        if rotate:
            # 3D interpolation with rotation as above reference
            try:
                htimes = tecxr.time.values
            except:
                htimes = np.array([float(tecxr.time)])  #in case of only one value
            # print(htimes)
            pretime = htimes[htimes <= acqtime].max()
            postime = htimes[htimes >= acqtime].min()
            #convet to timestamps to play with total_seconds
            pretime = pd.Timestamp(pretime)
            postime = pd.Timestamp(postime)
            
            lon0 = lon + (acqtime - pretime).total_seconds() / 86400 * 360. #let's play with seconds 24x60x60(seconds per day)
            lon1 = lon + (acqtime - postime).total_seconds() / 86400 * 360. #TODO or postime-acqtime?

            # #
            tec_val0 = float(tecxr.interp(time=pretime, lon=lon0, lat=lat, method='linear'))
            tec_val1 = float(tecxr.interp(time=postime, lon=lon1, lat=lat, method='linear'))
            
            tec = ((postime - acqtime).total_seconds() / (postime - pretime).total_seconds() * tec_val0
                   + (acqtime - pretime).total_seconds() / (postime - pretime).total_seconds() * tec_val1)
        else:
            # If rotation is NOT enabled, perform standard cubic interpolation
            tec = float(tecxr.interp(time=acqtime, lon=lon, lat=lat, method='cubic'))  
    return tec

In [7]:
def slant_ranges(frame, master, range2iono):
    """Computes slant range values and interpolates missing values.
    
    Args:
        frame (str): Frame ID of the LiCS frame (e.g., '116A_05207_252525').
        master (str): Master epoch of the frame (e.g., '20220901').
        range2iono (xarray.Dataset): Xarray dataset containing lon-lat coordinate info.
    
    Returns:
        xarray.Dataset: Interpolated slant range values.
    """
    
    # Define the path to the SLC parameter file
    SLCdir = os.path.join(os.environ['LiCSAR_procdir'], str(int(frame[:3])), frame, 'SLC', master)
    filepath = os.path.join(SLCdir, master + '.slc.par')
    
    orientation = frame[3]  # Extract orientation (A or D)

    # Read near, center, and far slant ranges
    near_range, center_range, far_range = None, None, None
    with open(filepath, 'r') as f:
        for line in f:
            if line.startswith('near_range_slc'):
                near_range = float(line.split()[1])
            elif line.startswith('center_range_slc'):
                center_range = float(line.split()[1])
            elif line.startswith('far_range_slc'):
                far_range = float(line.split()[1])

    # Ensure all values were found
    if None in [near_range, center_range, far_range]:
        raise ValueError(f"Missing range values in {filepath}")

    # Create a NaN-filled DataArray
    ds = range2iono.copy(deep=True)  # Preserve original shape and coordinates
    ds[:] = np.nan  # Fill with NaN
    
    # Compute scene center latitude
    scene_center_lat = range2iono.lat.mean().item()
    
    # Find the closest latitude index
    lat_idx = np.abs(range2iono.lat - scene_center_lat).argmin().item()
    
    # Extract longitude values along the center latitude
    lon_values = range2iono.lon.values
    data_values = range2iono.isel(lat=lat_idx).values  # Extract center latitude data
    
    # Mask NaN values and extract valid longitudes
    valid_lons = lon_values[~np.isnan(data_values)]
    
    if valid_lons.size > 0:
        leftmost_lon = np.min(valid_lons)
        rightmost_lon = np.max(valid_lons)
        center_lon = range2iono.lon.mean().item()
    else:
        raise ValueError("No valid non-NaN pixels found at the center latitude.")
    
    # Find longitude indices
    left_idx = np.abs(range2iono.lon - leftmost_lon).argmin().item()
    center_idx = np.abs(range2iono.lon - center_lon).argmin().item()
    right_idx = np.abs(range2iono.lon - rightmost_lon).argmin().item()

    ##consider about Ascending and desceding as the near and far range will be oriented.
    if orientation == 'A':
        valid_lon_indices = np.array([left_idx, center_idx, right_idx])
        valid_ranges = np.array([near_range, center_range, far_range])  # Left to right
    elif orientation == 'D':
        valid_lon_indices = np.array([right_idx, center_idx, left_idx])
        valid_ranges = np.array([near_range, center_range, far_range])  # Right to left
    
    
    ds[lat_idx, valid_lon_indices[0]] = valid_ranges[0]
    ds[lat_idx, valid_lon_indices[1]] = valid_ranges[1]
    ds[lat_idx, valid_lon_indices[2]] = valid_ranges[2]

    # Create interpolation function (linear)
    interp_func = interp1d(valid_lon_indices, valid_ranges, kind='linear', fill_value="extrapolate")

    # Interpolate NaNs along the longitude axis
    nan_mask = np.isnan(ds.isel(lat=lat_idx))  # Get NaN mask
    ds[lat_idx, nan_mask] = interp_func(np.where(nan_mask)[0])  # Interpolate missing values
    
    # Perform bivariate interpolation for full dataset (optional)
    ds = interpolate_nans_bivariate(ds)

    return ds


In [8]:
frame='116A_05207_252525'
epoch='20160112'
epoch='20160124'

# frame='021D_05266_252525'
# epoch='20160130'
metadir = os.path.join('/gws/nopw/j04/nceo_geohazards_vol1/public/LiCSAR_products', str(int(frame[:3])), frame, 'metadata')
metafile = os.path.join(metadir,'metadata.txt')
master=str(grep1line('master',metafile).split('=')[1])

In [9]:
str(frame[3])

'A'

In [10]:
##daz iono study!!!

In [11]:
alpha=0.85
source='code'
ionosampling = 10000  # m  --- by default, 20 km sampling should be ok?
inc=get_inc_frame(frame)
avg_incidence_angle = float(inc.mean())
# get hgt # no need anymore (2023/08)
metadir = os.path.join(os.environ['LiCSAR_public'],str(int(frame[:3])),frame,'metadata')
metafile = os.path.join(metadir,'metadata.txt')
scene_center_lon = float(inc.lon.mean())
scene_center_lat = float(inc.lat.mean())
#
center_time=grep1line('center_time',metafile).split('=')[1]
heading=float(grep1line('heading',metafile).split('=')[1])
try:
    centre_range_m = float(grep1line('centre_range_ok_m',metafile).split('=')[1]) # 2024: GAMMA had a bug wrongly informing on centre_range (may differ by 20 km or so!). fixed most of it
except:
    centre_range_m=float(grep1line('centre_range_m',metafile).split('=')[1])
#
master=str(grep1line('master',metafile).split('=')[1])
#
acqtime = pd.to_datetime(str(epoch) + 'T' + center_time)
# this is to get point between sat and scene centre
theta = np.radians(avg_incidence_angle)
wgs84 = nv.FrameE(name='WGS84')
Pscene_center = wgs84.GeoPoint(latitude=scene_center_lat, longitude=scene_center_lon, degrees=True)
burst_len = 7100*2.758277 #approx. satellite velocity on the ground 7100 [m/s] * burst_interval [s]
###### do the satg_lat, lon
azimuthDeg = heading - 90  # yes, azimuth is w.r.t. N (positive to E)
elevationDeg = 90 - avg_incidence_angle  # this is to get the avg sat altitude/range
slantRange = centre_range_m
hiono = 450
hiono = hiono * 1000  # m
resolution = get_resolution(inc, in_m=True)  # just mean avg in both lon, lat should be ok
# how large area is covered
lonextent = len(inc.lon) * resolution
# so what is the multilook factor?
mlfactorlon = round(len(inc.lon) / (lonextent / ionosampling))
latextent = len(inc.lat) * resolution
mlfactorlat = round(len(inc.lat) / (latextent / ionosampling))
#hgtml = hgt.coarsen({'lat': mlfactorlat, 'lon': mlfactorlon}, boundary='trim').mean()
incml = inc.coarsen({'lat': mlfactorlat, 'lon': mlfactorlon}, boundary='trim').mean() ##downsampling prevents unnecesarry high-frequency noise.
# get range towards iono single-layer in the path to the satellite, consider hgt
# range2iono = (hiono - hgtml) / np.cos(np.radians(incml))
# get range towards iono single-layer in the path to the satellite, do not consider hgt
range2iono = hiono / np.cos(np.radians(incml))  ##IPP to ground in the slant range direction
slantRanges=slant_ranges(frame, master, range2iono)
earth_radius = 6378160  # m
ionoxrA = incml.copy(deep=True)
ionoxrB = incml.copy(deep=True)
tecxr=get_vtec_from_code(acqtime, lat=0, lon=0, return_fullxr = True)
tecxr = alpha * tecxr
print('getting TEC values sampled by {} km.'.format(str(round(ionosampling / 1000))))

for i in range(len(range2iono.lat.values)):
    # print(str(i) + '/' + str(len(range2iono.lat.values)))
    for j in range(len(range2iono.lon.values)):
        if ~np.isnan(incml.values[i, j]):
            # theta = float(np.radians(incml.values[i, j]))
            eledeg = float(90 - incml.values[i, j])
            #ground_scene
            ilat_ground, ilon_ground = range2iono.lat.values[i], range2iono.lon.values[j]
            ##satellite scene, we need to consider satellite scene again for BOI
            x, y, z = aer2ecef(azimuthDeg, eledeg, slantRanges.values[i,j], ilat_ground, ilon_ground, 0) #scene_alt) ### this is wrt ellipsoid! ##TODO rather than the center slant range near center and far range can be added! IDK it's effect.
            satg_lat, satg_lon, sat_alt = ecef2latlonhei(x, y, z) ##satellite scene is changed becasue of the elevation degree change along the range direction
            sat_alt_km = round(sat_alt / 1000)
            Psatg = wgs84.GeoPoint(latitude=satg_lat, longitude=satg_lon, degrees=True)
            #then get A', B' ##squint angle in azimuth direction lead that shift.
            PsatgA, _azimuth = Psatg.displace(distance=burst_len/2, azimuth=heading-180, method='ellipsoid', degrees=True)
            PsatgB, _azimuth = Psatg.displace(distance=burst_len/2, azimuth= heading, method='ellipsoid', degrees=True)
            
            ##IPP scene, 
            x, y, z = aer2ecef(azimuthDeg, eledeg, range2iono.values[i, j], ilat_ground, ilon_ground, 0) #range should be in meter
            ippg_lat, ippg_lon, ipp_alt = ecef2latlonhei(x, y, z)
            Pippg = wgs84.GeoPoint(latitude=ippg_lat, longitude=ippg_lon, degrees=True)
            #then get A', B'
            PippAt, _azimuth = Pippg.displace(distance=burst_len, azimuth=heading-180, method='ellipsoid', degrees=True) #We extend burst_len to cover the intersection area. theorically burst_length/2 ideal
            PippBt, _azimuth = Pippg.displace(distance=burst_len, azimuth= heading, method='ellipsoid', degrees=True)

            ##intersection, to make sure IPP coordinates found
            path_ipp = nv.GeoPath(PippAt, PippBt)
            Pscene_center = wgs84.GeoPoint(latitude=ilat_ground, longitude=ilon_ground, degrees=True)
            path_scene_satgA = nv.GeoPath(Pscene_center, PsatgA)
            path_scene_satgB = nv.GeoPath(Pscene_center, PsatgB)
            # these two points are the ones where we should get TEC
            PippA = path_ipp.intersect(path_scene_satgA).to_geo_point()
            PippB = path_ipp.intersect(path_scene_satgB).to_geo_point()

            if source=='code':
                # print('code selected')
                ionoijA = get_vtec_from_tecxr(tecxr, acqtime, PippA.latitude_deg, PippA.longitude_deg)
                ionoijB = get_vtec_from_tecxr(tecxr, acqtime, PippB.latitude_deg, PippB.longitude_deg)
            else:
                # print('iri selected')
                ionoijA = get_tecs(PippA.latitude_deg, PippA.longitude_deg, sat_alt_km, [acqtime], False, source=source)[0]  ##TODO ask Milan why we set the sat_alt_km rather than ipp_alt when we apply IRI model? it select IPP itself?
                ionoijB = get_tecs(PippB.latitude_deg, PippB.longitude_deg, sat_alt_km, [acqtime], False, source=source)[0]

            #VTEC2STEC        
            theta = float(np.radians(incml.values[i, j]))
            sin_thetaiono = earth_radius / (earth_radius + hiono) * np.sin(theta)
            ionoxrA.values[i, j] = ionoijA / np.sqrt(1 - sin_thetaiono ** 2)
            ionoxrB.values[i, j] = ionoijB / np.sqrt(1 - sin_thetaiono ** 2)
        
        # # Let's make it a bit more robust 
        # ionoxr_dict = {'A': ionoxrA, 'B': ionoxrB}
        # outputxr_dict = {}

        # for key in ['A', 'B']:
        #     ionoxr_dict[key] = interpolate_nans_bivariate(ionoxr_dict[key])
        #     ionoxr_dict[key] = ionoxr_dict[key].interp_like(inc, method='linear', kwargs={"bounds_error": False, "fill_value": None})

        #     if np.max(np.isnan(ionoxr_dict[key].values)):
        #         ionoxr_dict[key] = interpolate_nans_bivariate(ionoxr_dict[key])  # Not the best, but needed

        #     outputxr_dict[key] = ionoxr_dict[key]


getting TEC values sampled by 10 km.


In [12]:
tecphase.interp_like

NameError: name 'tecphase' is not defined

In [ ]:
ionoxrA.plot()

In [ ]:
ionoxrB.plot()

In [ ]:
(ionoxrA-ionoxrB).plot()

In [ ]:
1.0526707446e+17-1.05052739119e+17

In [ ]:
ionoxrA.plot()

In [ ]:
ionoxrB.plot()

In [ ]:
(ionoxrA-ionoxrB).plot()

In [ ]:
#################################

In [ ]:
alpha=0.85
tecxr=get_vtec_from_code(acqtime, lat=0, lon=0, return_fullxr = True)
tecxr = alpha * tecxr

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr

# Define the figure and axes (4 rows, 6 columns)
fig, axes = plt.subplots(16, 6, figsize=(18, 48))

# Adjust layout
fig.suptitle("Global TEC Maps (Total Electron Content) at Different Hours", fontsize=16)

# Determine vmin and vmax for a consistent color scale
vmin = tecxr.min().values
vmax = tecxr.max().values

# Loop through time steps and plot on subplots
for i, ax in enumerate(axes.flat):
    if i < len(tecxr.time):
        # Select the TEC map at time `i`
        tecxr_selected = tecxr.sel(time=tecxr.time[i])

        # Plot on the current axis with fixed vmin/vmax
        im = tecxr_selected.plot(ax=ax, cmap='jet', add_colorbar=False, vmin=vmin, vmax=vmax)
        
        # Set title as time step
        # ax.set_title(f"Time: {int(tecxr.time[i].values)} UTC")
        
        # Remove axis labels for cleaner visualization
        ax.set_xlabel("")
        ax.set_ylabel("")
    else:
        ax.axis("off")  # Hide unused subplots if any

# Create a single colorbar
cbar_ax = fig.add_axes([0.92, 0.2, 0.02, 0.6])  # [left, bottom, width, height]
fig.colorbar(im, cax=cbar_ax, label="TEC Units (TECU)")

# Adjust spacing
plt.tight_layout(rect=[0, 0, 0.9, 0.95])  # Leave space for the colorbar
plt.show()
